# Vídeos - Jaqueta - Segmentação do Coral-Sol nos vídeos

Autor:  Viviane da Rosa Sommer

Entrega: 14/02/2025

## Objetivo:
Notebook para fazer a inferência com o modelo MSOV3 de Segmentação, no conjunto de vídeos de Jaqueta.

## Importações Necessárias

In [ ]:
!pip install opencv-python-headless
!pip install ultralytics
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import torchvision
import torch
from tqdm.notebook import tqdm
import pandas as pd
import math
import os
import glob

## Declaração de Constantes e Modelos

In [ ]:
BATCH_SIZE = 20 
INFERENCE_INTERVAL = 0
CORAL_CLASS_ID = 0
coral_model = YOLO("../Imagens de Mergulho - Modelo de Segmentação/runs/segment/train/weights/best.pt")

IN_PATH = "Clips-50-60"
OUT_PATH = "Clips-50-60-Resultado"
CSV_FOLDER = ""
PLOT_FOLDER = ""

os.mkdir(OUT_PATH)
os.mkdir(CSV_FOLDER)
os.mkdir(PLOT_FOLDER)

## Funções Necessárias

In [ ]:
def process_results(results: list) -> any:
    """
    Processes the results obtained from the model, returning the first valid result with masks.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result containing masks, or None if no valid result is found.
    """
    for result in results:
        if hasattr(result, 'masks') and result.masks is not None:
            return result
    return None


def prediction_coral(img: np.ndarray) -> tuple:
    """
    Predicts coral objects using the coral model and returns individual masks and their scores.

    Parameters:
        img (np.ndarray): Input image for prediction.

    Returns:
        tuple: A tuple containing:
            - masks (list): List of binary masks for the predicted coral regions.
            - scores (list): List of scores for each predicted coral region.
    """
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)

    masks_list = []
    scores_list = []

    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        if len(coral_indices) > 0:
            coral_boxes = boxes[coral_indices]
            coral_masks = masks[coral_indices]
            coral_scores = scores[coral_indices]

            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            for idx in nms_indices:
                mask_np = coral_masks[idx].int().cpu().numpy() * 255
                masks_list.append(mask_np.astype(np.uint8))
                scores_list.append(coral_scores[idx].item())

    return masks_list, scores_list


def process_video_and_save_scores(video_path: str, output_path: str) -> tuple:
    """
    Processes the video frame by frame, performs inference on every frame, and saves the annotated video.

    Parameters:
        video_path (str): Path to the input video file.
        output_path (str): Path to the output video file.

    Returns:
        tuple: Two lists, one with times and one with average scores per frame.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return [], []

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    times = []
    average_scores = []

    for frame_count in tqdm(range(total_frames), desc="Processing Video"):
        ret, frame = cap.read()
        if not ret:
            break

        timestamp = frame_count / fps
        times.append(timestamp)

        cropped_frame, top, bottom = crop_frame(frame)
        masks_list, scores_list = prediction_coral(cropped_frame)
        average_score = sum(scores_list) / len(scores_list) if scores_list else 0.0
        average_scores.append(average_score)

        frame_with_mask = overlay_masks(frame, masks_list, top, bottom, scores_list)
        out.write(frame_with_mask)

    cap.release()
    out.release()

    return times, average_scores


def crop_frame(frame: np.ndarray) -> tuple:
    """
    Crops the frame to focus on the central region.

    Parameters:
        frame (np.ndarray): The frame to crop.

    Returns:
        tuple: A tuple containing the cropped frame and the top and bottom crop coordinates.
    """
    height, _, _ = frame.shape
    mid = height // 2
    top = max(0, mid - int(0.34 * height))
    bottom = min(height, mid + int(0.34 * height))
    cropped_frame = frame[top:bottom, :]
    return cropped_frame, top, bottom


def overlay_masks(frame: np.ndarray, masks_list: list, top: int, bottom: int, scores_list: list) -> np.ndarray:
    """
    Overlays the masks on the original frame and adds annotations.

    Parameters:
        frame (np.ndarray): The original video frame.
        masks_list (list): List of binary masks to overlay.
        top (int): The top crop coordinate.
        bottom (int): The bottom crop coordinate.
        scores_list (list): List of scores for each mask.

    Returns:
        np.ndarray: The frame with the masks overlaid and annotations.
    """
    if not masks_list:
        center_y = frame.shape[0] // 2
        cv2.putText(frame, "No corals detected", (10, center_y), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        return frame

    for mask, score in zip(masks_list, scores_list):
        mask_resized = cv2.resize(mask, (frame.shape[1], bottom - top))
        contours, _ = cv2.findContours(mask_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(frame[top:bottom, :], contours, -1, (0, 0, 255), 2)
        for contour in contours:
            x, y, _, _ = cv2.boundingRect(contour)
            score_text = f'{score:.2f}'
            cv2.putText(frame, score_text, (x, y + top - 5), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 1, cv2.LINE_AA)

    return frame


def save_scores_to_csv(times: list, average_scores: list, csv_path: str, frame_interval: int = 30) -> None:
    """
    Saves the times and scores to a CSV file.

    Parameters:
        times (list): List of times corresponding to the frames.
        average_scores (list): List of scores for each frame.
        csv_path (str): Path to the output CSV file.
        frame_interval (int): The interval of frames where inference was performed (default: 30).
    """
    try:
        if len(times) > len(average_scores):
            times = times[:len(average_scores)]
        elif len(average_scores) > len(times):
            average_scores = average_scores[:len(times)]

        df = pd.DataFrame({'Time (seconds)': times, 'Average Score': average_scores})
        df['Time (HH:MM:SS)'] = pd.to_datetime(df['Time (seconds)'], unit='s').dt.strftime('%H:%M:%S')
        df['Time (seconds)'] = df['Time (seconds)'].apply(math.floor)
        df = df.groupby(['Time (seconds)', 'Time (HH:MM:SS)'], as_index=False).max()
        df.to_csv(csv_path, index=False)
        print(f"Results saved to: {csv_path}")

    except Exception as e:
        print(f"Error saving CSV: {e}")


def plot_from_csv_max_score_per_second(csv_path: str, output_plot_path: str, video_name: str) -> None:
    """
    Reads a CSV file, selects the maximum score for each second, and generates a line plot.

    Parameters:
        csv_path (str): Path to the input CSV file.
        output_plot_path (str): Path to save the plot as an image file.
        video_name (str): Name of the video to display in the plot title.
    """
    df = pd.read_csv(csv_path)

    if "Time (HH:MM:SS)" not in df.columns or "Average Score" not in df.columns:
        print(f"Invalid CSV or missing columns: {csv_path}")
        return

    times = df["Time (HH:MM:SS)"]
    scores = df["Average Score"]

    plt.figure(figsize=(16, 8))
    plt.plot(times, scores, color='b', linewidth=2, label="Max Score per Second")
    plt.xlabel('Time (HH:MM:SS)', fontsize=14)
    plt.ylabel('Average Score', fontsize=14)
    plt.title(f'Maximum Average Score Per Second\nVideo: {video_name}', fontsize=16)
    plt.ylim(0, 1)
    spacing = max(1, len(times) // 10)
    plt.xticks(range(0, len(times), spacing), times[::spacing], rotation=45, fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(output_plot_path)
    plt.close()
    print(f"Plot saved to: {output_plot_path}")


def merge_videos_with_titles_efficient(folder_path: str, output_path: str) -> None:
    """
    Efficiently merges all video files in the specified folder into a single video.

    Parameters:
        folder_path (str): Path to the folder containing the video files.
        output_path (str): Path to save the merged video file.
    """
    try:
        video_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
        video_files.sort()

        if not video_files:
            print("No videos found in the specified folder.")
            return

        sample_video = cv2.VideoCapture(os.path.join(folder_path, video_files[0]))
        fps = int(sample_video.get(cv2.CAP_PROP_FPS))
        width = int(sample_video.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(sample_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        sample_video.release()

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        total_frames = 0
        for video_file in video_files:
            video_path = os.path.join(folder_path, video_file)
            cap = cv2.VideoCapture(video_path)
            total_frames += int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            total_frames += fps * 2
            cap.release()

        with tqdm(total=total_frames, desc="Processing Videos", unit="frame") as pbar:
            for video_file in video_files:
                video_path = os.path.join(folder_path, video_file)
                video_name = os.path.splitext(video_file)[0]
                transition_frame = create_transition_frame(video_name, width, height)
                for _ in range(fps * 2):
                    out.write(transition_frame)
                    pbar.update(1)

                cap = cv2.VideoCapture(video_path)
                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    out.write(frame)
                    pbar.update(1)
                cap.release()

        out.release()
        print(f"Final video saved to: {output_path}")

    except Exception as e:
        print(f"Error merging videos: {e}")


def create_transition_frame(text: str, width: int, height: int) -> np.ndarray:
    """
    Creates a black frame with centered white text as a transition screen.

    Parameters:
        text (str): Text to display on the transition screen.
        width (int): Width of the frame.
        height (int): Height of the frame.

    Returns:
        np.ndarray: The generated frame with text.
    """
    max_width = width - 40
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        if cv2.getTextSize(current_line + " " + word, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)[0][0] <= max_width:
            current_line += " " + word
        else:
            lines.append(current_line)
            current_line = word

    lines.append(current_line)

    frame = np.zeros((height, width, 3), dtype=np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    color = (255, 255, 255)
    thickness = 2
    total_text_height = sum([cv2.getTextSize(line, font, font_scale, thickness)[0][1] for line in lines])
    y_offset = (height - total_text_height) // 2

    for line in lines:
        text_size = cv2.getTextSize(line, font, font_scale, thickness)[0]
        text_x = (width - text_size[0]) // 2
        cv2.putText(frame, line, (text_x, y_offset), font, font_scale, color, thickness)
        y_offset += text_size[1]

    return frame

## Segmenta o cora-sol, frame por frame de cada vídeo na pasta. 
### Cria também um CSV com o maior score por segundo e o Plot de Score por Tempo.

In [ ]:
for filename in glob.glob(os.path.join(IN_PATH, "*")):
    output_path = os.path.join(OUT_PATH, os.path.basename(filename))
    csv_path = os.path.join(CSV_FOLDER, os.path.splitext(os.path.basename(filename))[0] + ".csv")
    plot_path = os.path.join(PLOT_FOLDER, os.path.splitext(os.path.basename(filename))[0] + ".png")
    video_name = os.path.basename(filename) 
    print(f"Processando '{filename}'...")
    times, average_scores = process_video_and_save_scores(filename, output_path)
    save_scores_to_csv(times, average_scores, csv_path) 
    plot_from_csv_max_score_per_second(csv_path, plot_path, video_name)  
    print(f"Processamento de '{filename}' concluído!")

## Junta os clips em um só vídeo, para facilitar a visualização dos Falsos Positivos

In [ ]:
merge_videos_with_titles_efficient(OUT_PATH, OUT_PATH + ".mp4")